# [Sensitivity Analysis] The Over-Parameterization (KAN-TabNet) | Step LR | Forest Cover

**Optimized Reference Configuration:** `n_d=20, n_a=20`, `kan_grid_size=5`, `kan_spline_order=3`, `initial_lr=0.002`

### Methodological Context: The Control Variables
To precisely isolate the effects of spline resolution scaling on high-dimensional mixed data, this sensitivity analysis uses the exact same massive routing pipeline (`20x20`), cubic spline order (`k=3`), StepLR schedule, and initial learning rate (`0.002`) as our optimized reference configuration.

By fixing the structural routing capacity and thermodynamic optimization environment, we strictly guarantee that any performance variations observed in this notebook are purely the result of expanding the internal grid intervals ($G$) of the KAN layers.

### The Hypothesis
In this study, we evaluate a highly expanded grid resolution (`kan_grid_size=10`). Because the parameter count of a KAN layer scales quadratically with pipeline width and linearly with grid size, applying $G=10$ to a massive $20 \times 20$ pipeline inflates the network's overall parameter count astronomically.

While the Forest Cover dataset requires flexibility to map its geography, the majority of its 54 features are highly sparse, one-hot encoded categorical variables (such as 40 distinct soil types). We hypothesize that providing this much mathematical resolution ($G=10$) introduces severe diminishing returns and likely triggers active over-parameterization. Specifically, the network will use the excess B-spline control points to overfit the noise within the sparse categorical data rather than learning generalizable topographical rules, ultimately validating $G=5$ as the optimal complexity ceiling.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier

dataset = fetch_covtype()

X = dataset.data
# CovType is 1-indexed (1 to 7); PyTorch expects 0-indexed labels
y = dataset.target - 1

# divide dataset into 80% train, 20% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

# divide temp into 10% validation and 10% test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)

print(f"Train shape: {X_train.shape}")
print(f"Valid shape: {X_valid.shape}")
print(f"Test shape:  {X_test.shape}")

Train shape: (464809, 54)
Valid shape: (58101, 54)
Test shape:  (58102, 54)


### Model Training

In [5]:
MAX_EPOCHS = 1000

In [6]:
# Hyperparameters from original paper.
# Notes:
# - momentum is 1 - 0.7 (paper m_B) to align with PyTorch's BatchNorm API.
# - scheduler step_size of 18 epochs approximates the paper's 500 iterations
#   (based on a batch size of 16384 and ~464k training samples).
paper_config = {
    'n_steps': 5,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-4,
    'momentum': 0.3,
    'optimizer_fn': torch.optim.Adam,
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'scheduler_params': dict(step_size=18, gamma=0.95),
    'mask_type': 'sparsemax'
}

clf_kan = TabNetClassifier(
    n_d=20,
    n_a=20,
    **paper_config,
    clip_value=2.,
    optimizer_params=dict(lr=0.002),
    use_kan=True,
    kan_grid_size=10,
    kan_spline_order=3,
    seed=seed,
    verbose=50
)

In [7]:
# Hyperparameters from original paper.
paper_fit_config = {
    'batch_size': 16384,
    'virtual_batch_size': 512,
}

clf_kan.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=100,
    num_workers=0,
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 1.99237 | valid_accuracy: 0.45431 |  0:00:09s
epoch 50 | loss: 0.29648 | valid_accuracy: 0.88931 |  0:07:11s
epoch 100| loss: 0.18661 | valid_accuracy: 0.93253 |  0:14:14s
epoch 150| loss: 0.21822 | valid_accuracy: 0.9178  |  0:21:16s
epoch 200| loss: 0.14519 | valid_accuracy: 0.94785 |  0:28:17s
epoch 250| loss: 0.14228 | valid_accuracy: 0.95072 |  0:35:22s
epoch 300| loss: 0.10263 | valid_accuracy: 0.96019 |  0:42:29s
epoch 350| loss: 0.09635 | valid_accuracy: 0.96157 |  0:49:40s
epoch 400| loss: 0.09276 | valid_accuracy: 0.96157 |  0:56:52s
epoch 450| loss: 0.09055 | valid_accuracy: 0.96179 |  1:04:02s
epoch 500| loss: 0.08773 | valid_accuracy: 0.96244 |  1:11:14s
epoch 550| loss: 0.07487 | valid_accuracy: 0.96547 |  1:18:26s
epoch 600| loss: 0.07001 | valid_accuracy: 0.96647 |  1:25:43s
epoch 650| loss: 0.06652 | valid_accuracy: 0.96701 |  1:33:04s
epoch 700| loss: 0.0643  | valid_accuracy: 0.96812 |  1:40:26s
epoch 750| loss: 0.05927 | valid_accuracy: 0.96895 |  1

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_kan.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 698,936


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_kan.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.97059


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/forest_cover/tables'
MODELS_DIR = f'{BASE_DIR}/results/forest_cover/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "KAN-TabNet",
    "dataset": "Forest Cover",
    "parameters": param_count,
    "scheduler": "StepLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_kan.best_epoch,
    "training_history": clf_kan.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/06_kan_sensitivity_grid_10_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_kan.save_model(f'{MODELS_DIR}/06_kan_sensitivity_grid_10_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/forest_cover/tables/06_kan_sensitivity_grid_10_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/forest_cover/models/06_kan_sensitivity_grid_10_698k.zip
